# 1. Installazione ed Autenticazione

In [ ]:
# Installiamo la libreria di Google per Fogli
!pip install gspread

# Importiamo le librerie necessarie
import gspread
import pandas as pd
from google.colab import auth

# Autentichiamo l'utente
auth.authenticate_user()
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)

print("✅ Autenticazione completata.")

✅ Autenticazione completata.


# 2. Connessione e Lettura Dati

In [ ]:
# Apriamo il Google Sheet per NOME.
# Assicurati che il nome sia ESATTO.
try:
    ss = gc.open("Fanta CY")
    print("✅ File 'Fanta CY' aperto con successo.")

    # Leggiamo i fogli di input
    db_ws = ss.worksheet("DB_Giocatori")
    voti_ws = ss.worksheet("Input_Voti")
    form_ws = ss.worksheet("Form_Responses")

    # Trasformiamo i fogli in DataFrame Pandas
    # get_all_records() legge i dati usando la prima riga come intestazione
    df_db = pd.DataFrame(db_ws.get_all_records())
    df_voti = pd.DataFrame(voti_ws.get_all_records())
    df_form = pd.DataFrame(form_ws.get_all_records())

    print("DB Giocatori Caricato:", len(df_db), "giocatori")
    print("Input Voti Caricato:", len(df_voti), "voti")
    print("Formazioni Ricevute:", len(df_form), "formazioni")

except Exception as e:
    print(f"🛑 ERRORE: Impossibile aprire il file o i fogli. Controlla i nomi.")
    print(e)

In [14]:
# --- CELLA DI DEBUG ---
# Esegui questa cella DOPO la Cella 2 per vedere i nomi esatti

if 'df_form' in locals():
  print("✅ Nomi esatti delle colonne lette da 'Form_Responses':")
  print(list(df_form.columns))
else:
  print("🛑 Errore: Esegui prima la Cella 2 per caricare 'df_form'.")

✅ Nomi esatti delle colonne lette da 'Form_Responses':
['Informazioni cronologiche', 'Indirizzo email', 'Nome', 'Cognome', 'Stanza', 'Scelta Portiere', 'Scelta Difensori', 'Scelta Centrocampisti', 'Scelta Attaccante', 'Nickname Squadra']


# 3. Preparazione Dati

In [15]:
# Pulizia e Calcolo Voti
print("Inizio preparazione dati...")

# Lista delle colonne bonus/malus (devono corrispondere al foglio!)
colonne_bm = [
    'Goal_Fatto', 'Goal_Subito', 'Falli', 'Giallo', 'Rosso',
    'Squadra_Vince', 'Squadra_Perde', 'Porta_Inviolata', 'MVP',
    'Rigore_Parato', 'Rigore_Sbagliato', 'Autogoal'
]

# Convertiamo le colonne B/M in numeri (se vuote, diventano 0)
for col in colonne_bm:
    if col in df_voti:
        df_voti[col] = pd.to_numeric(df_voti[col], errors='coerce').fillna(0)
    else:
        print(f"Attenzione: Colonna '{col}' non trovata in 'Input_Voti'")

# Calcoliamo il punteggio finale per ogni giocatore
# (Se un giocatore non ha voti, riceverà 0, non 6)
df_voti['Punteggio_Finale'] = 6 + \
    (df_voti.get('Goal_Fatto', 0) * 3) - \
    (df_voti.get('Goal_Subito', 0) * 1) - \
    (df_voti.get('Falli', 0) * 0.5) - \
    (df_voti.get('Giallo', 0) * 1) - \
    (df_voti.get('Rosso', 0) * 2) + \
    (df_voti.get('Squadra_Vince', 0) * 0.5) - \
    (df_voti.get('Squadra_Perde', 0) * 0.5) + \
    (df_voti.get('Porta_Inviolata', 0) * 1) + \
    (df_voti.get('MVP', 0) * 1) + \
    (df_voti.get('Rigore_Parato', 0) * 3) - \
    (df_voti.get('Rigore_Sbagliato', 0) * 3) - \
    (df_voti.get('Autogoal', 0) * 3)

# 3.2. Creazione Mappe (Dizionari)
# Convertiamo 'Costo' in numero per sicurezza
df_db['Costo'] = pd.to_numeric(df_db['Costo'], errors='coerce').fillna(0)

# Mappa Nome -> Costo
# Esempio: {'Maci': 10, 'Giangiovanni': 15}
prezzi_map = df_db.set_index('Nome_Giocatore')['Costo'].to_dict()

# Mappa Nome -> Punteggio_Finale
# Esempio: {'Maci': 6.5, 'Giangiovanni': 9.0}
punti_map = df_voti.set_index('Nome_Giocatore')['Punteggio_Finale'].to_dict()

print("✅ Mappe prezzi e punti create.")

Inizio preparazione dati...
✅ Mappe prezzi e punti create.


## 3.1 Versione Svincolati

In [16]:
# --- 3. PREPARAZIONE DATI (Versione con 'sv') ---

# ⚠️ IMPOSTAZIONE MANUALE ⚠️
# Definisci la colonna che userai per il flag 'sv' (Senza Voto)
# Assicurati che il nome sia ESATTO.
COLONNA_FLAG_SV = 'Goal_Fatto'

print("Inizio preparazione dati (con logica 'sv')...")

# 3.1. Pulizia e Calcolo Voti
colonne_bm = [
    'Goal_Fatto', 'Goal_Subito', 'Falli', 'Giallo', 'Rosso',
    'Squadra_Vince', 'Squadra_Perde', 'Porta_Inviolata', 'MVP',
    'Rigore_Parato', 'Rigore_Sbagliato', 'Autogoal'
]

# Copia per evitare 'SettingWithCopyWarning'
df_voti_pulito = df_voti.copy()

# 1. Identifica chi ha 'sv'
# Controlliamo che la colonna esista
if COLONNA_FLAG_SV not in df_voti_pulito.columns:
    raise ValueError(f"ERRORE: La colonna '{COLONNA_FLAG_SV}' non è in Input_Voti.")

# .astype(str) è fondamentale per comparare con 'sv'
# .str.lower() gestisce se scrivi 'sv' o 'SV'
is_sv = df_voti_pulito[COLONNA_FLAG_SV].astype(str).str.lower() == 'sv'

# 2. Inizializza Punteggio Finale
# Tutti partono da 6 (voto politico)
df_voti_pulito['Punteggio_Finale'] = 6.0

# 3. Converti TUTTE le colonne bonus/malus in numeri
# 'sv' diventerà NaN (che poi .fillna(0) trasformerà in 0)
# Gli spazi vuoti diventeranno 0
for col in colonne_bm:
    df_voti_pulito[col] = pd.to_numeric(df_voti_pulito[col], errors='coerce').fillna(0)

# 4. Calcola i punteggi (sovrascrive il 6)
# Questo calcolo ora è basato su 0 per 'sv', che è corretto.
df_voti_pulito['Punteggio_Finale'] = 6.0 + \
    (df_voti_pulito.get('Goal_Fatto', 0) * 3) - \
    (df_voti_pulito.get('Goal_Subito', 0) * 1) - \
    (df_voti_pulito.get('Falli', 0) * 0.5) - \
    (df_voti_pulito.get('Giallo', 0) * 1) - \
    (df_voti_pulito.get('Rosso', 0) * 2) + \
    (df_voti_pulito.get('Squadra_Vince', 0) * 0.5) - \
    (df_voti_pulito.get('Squadra_Perde', 0) * 0.5) + \
    (df_voti_pulito.get('Porta_Inviolata', 0) * 1) + \
    (df_voti_pulito.get('MVP', 0) * 1) + \
    (df_voti_pulito.get('Rigore_Parato', 0) * 3) - \
    (df_voti_pulito.get('Rigore_Sbagliato', 0) * 3) - \
    (df_voti_pulito.get('Autogoal', 0) * 3)

# 5. Applica il flag 'sv'
# Sovrascrivi il punteggio (che era 6) e mettilo a 0
# usiamo .loc per assicurarci di modificare il DataFrame originale
df_voti_pulito.loc[is_sv, 'Punteggio_Finale'] = 0.0

# 3.2. Creazione Mappe (Dizionari)
df_db['Costo'] = pd.to_numeric(df_db['Costo'], errors='coerce').fillna(0)

# Mappa Nome -> Costo
prezzi_map = df_db.set_index('Nome_Giocatore')['Costo'].to_dict()

# Mappa Nome -> Punteggio_Finale
# Questa mappa ora contiene '0.0' per i giocatori 'sv'
punti_map = df_voti_pulito.set_index('Nome_Giocatore')['Punteggio_Finale'].to_dict()

print("✅ Mappe prezzi e punti create (giocatori 'sv' avranno 0 punti).")

Inizio preparazione dati (con logica 'sv')...
✅ Mappe prezzi e punti create (giocatori 'sv' avranno 0 punti).


# 4. Elaborazione Formazioni


In [ ]:
# ⚠️ CONFIGURAZIONE OBBLIGATORIA ⚠️
# Ho aggiunto 'email' e 'nickname'. Controlla che i nomi siano corretti!
COLONNE_MAP = {
    'email': 'Indirizzo email', # <-- CHIAVE UNIVOCA
    'nome': 'Nome',
    'cognome': 'Cognome',
    'nickname': 'Nickname Squadra', # <-- NUOVA COLONNA
    'portiere': 'Scelta Portiere',
    'difensori': 'Scelta Difensori',
    'centrocampisti': 'Scelta Centrocampisti',
    'attaccante': 'Scelta Attaccante'
}

# Imposta il tuo budget
BUDGET_MASSIMO = 100 # ⚠️ Modifica questo con il tuo budget!

# Questo comando dice: "Tieni solo le righe dove la colonna email non è vuota"
df_form = df_form[df_form[COLONNE_MAP['email']].astype(str).str.strip() != ""].copy()

# Lista per salvare i risultati finali
risultati_giornata = []

# Funzione per pulire il nome. "Maci (10)" -> "Maci"
def pulisci_nome(nome_raw):
    if not isinstance(nome_raw, str):
        return ""
    return nome_raw.split(' (')[0].strip()

print(f"Inizio elaborazione di {len(df_form)} formazioni...")
print(f"Budget massimo impostato: {BUDGET_MASSIMO}")

for index, riga in df_form.iterrows():

    squadra_raw = []
    costo_totale = 0
    punteggio_totale = 0
    status = "VALIDA"
    squadra_pulita = []

    # Identificativo utente
    try:
        # La chiave univoca ora è l'email
        email_utente = riga[COLONNE_MAP['email']]
        nome_utente = f"{riga[COLONNE_MAP['nome']]} {riga[COLONNE_MAP['cognome']]}"
        nickname_squadra = riga[COLONNE_MAP['nickname']]

        # Gestiamo nickname vuoti
        if not nickname_squadra or pd.isna(nickname_squadra):
            nickname_squadra = nome_utente # Fallback sul nome utente

    except KeyError as e:
        print(f"ERRORE CRITICO: Impossibile leggere colonna {e}. Fermo elaborazione riga.")
        continue # Salta questa riga
    except Exception as e:
        email_utente = f"Errore_Email_{index}"
        nome_utente = f"Errore_Nome_{index}"
        nickname_squadra = "Errore_Nickname"
        print(f"Errore lettura metadati utente: {e}")

    try:
        # 1. RACCOLTA E PULIZIA GIOCATORI
        p_raw = riga[COLONNE_MAP['portiere']]
        a_raw = riga[COLONNE_MAP['attaccante']]
        squadra_raw.extend([p_raw, a_raw])

        dif_raw_list = riga[COLONNE_MAP['difensori']].split(',')
        cen_raw_list = riga[COLONNE_MAP['centrocampisti']].split(',')

        squadra_raw.extend([d.strip() for d in dif_raw_list])
        squadra_raw.extend([c.strip() for c in cen_raw_list])

        squadra_pulita = [pulisci_nome(g) for g in squadra_raw if pulisci_nome(g)]

        # 2. VALIDAZIONE
        if len(squadra_pulita) != 7:
            status = f"INVALIDA ({len(squadra_pulita)} giocatori trovati, non 7)"
        else:
            for nome in squadra_pulita:
                costo_giocatore = prezzi_map.get(nome)
                punti_giocatore = punti_map.get(nome, 6)

                if costo_giocatore is None:
                    status = f"INVALIDA (Giocatore '{nome}' non trovato nel DB)"
                    costo_totale = 0
                    break

                costo_totale += costo_giocatore
                punteggio_totale += punti_giocatore

            if status == "VALIDA" and costo_totale > BUDGET_MASSIMO:
                status = f"INVALIDA (Budget sforato: {costo_totale})"

        # 3. Assegnazione Punteggio
        if status != "VALIDA":
            punteggio_totale = 0

        risultati_giornata.append({
            'Email': email_utente, # <-- NUOVA CHIAVE
            'Utente': nome_utente, # Nome e Cognome
            'Nickname': nickname_squadra, # Nickname
            'Punteggio': round(punteggio_totale, 2),
            'Costo': costo_totale,
            'Status': status,
            'Squadra': ", ".join(squadra_pulita)
        })

    except Exception as e:
        risultati_giornata.append({
            'Email': email_utente,
            'Utente': nome_utente,
            'Nickname': nickname_squadra,
            'Punteggio': 0, 'Costo': 0,
            'Status': f"ERRORE LETTURA (Cella vuota? Dettaglio: {e})",
            'Squadra': ""
        })

print("✅ Elaborazione terminata.")

# Convertiamo i risultati in un DataFrame
df_risultati = pd.DataFrame(risultati_giornata)
# Impostiamo l'Email come indice per i controlli (anche se non strettamente nec.)
df_risultati.set_index('Email', inplace=True)

print("\n--- RISULTATI GIORNATA (Top 10) ---")
print(df_risultati.sort_values(by="Punteggio", ascending=False).head(10))

print("\n--- FORMAZIONI CON ERRORI O INVALIDE ---")
print(df_risultati[df_risultati['Status'] != 'VALIDA'])

# Riportiamo l'Email a colonna per la Cella 5
df_risultati.reset_index(inplace=True)

# 5A. Trascrizione dei risultati nella Classifica (provvisorio)

In [19]:
"""
try:
    # Apri il foglio "Classifica"
    classifica_ws = ss.worksheet("Classifica")

    # Pulisci il foglio (opzionale, ma consigliato)
    classifica_ws.clear()

    # Converti il DataFrame in una lista di liste (incluso l'header)
    output_data = [df_risultati.columns.values.tolist()] + df_risultati.values.tolist()

    # Scrivi i dati sul foglio
    classifica_ws.update('A1', output_data, value_input_option='USER_ENTERED')

    print(f"\n✅ Dati scritti con successo sul foglio 'Classifica'!")
    print(f"Link: {ss.url}")

except Exception as e:
    print(f"\n🛑 ERRORE: Impossibile scrivere sul foglio 'Classifica'.")
    print(e)
"""

'\ntry:\n    # Apri il foglio "Classifica"\n    classifica_ws = ss.worksheet("Classifica")\n\n    # Pulisci il foglio (opzionale, ma consigliato)\n    classifica_ws.clear()\n\n    # Converti il DataFrame in una lista di liste (incluso l\'header)\n    output_data = [df_risultati.columns.values.tolist()] + df_risultati.values.tolist()\n\n    # Scrivi i dati sul foglio\n    classifica_ws.update(\'A1\', output_data, value_input_option=\'USER_ENTERED\')\n\n    print(f"\n✅ Dati scritti con successo sul foglio \'Classifica\'!")\n    print(f"Link: {ss.url}")\n\nexcept Exception as e:\n    print(f"\n🛑 ERRORE: Impossibile scrivere sul foglio \'Classifica\'.")\n    print(e)\n'

# 5B. Aggiornamento Classifica Storica

In [ ]:
# --- 5. AGGIORNAMENTO CLASSIFICA STORICA (Versione 5.5 - Ordine Invertito) ---

# 5.1. Chiedi il nome (prefisso) per la nuova giornata
nome_giornata_prefix = input("Inserisci il PREFISSO per questa giornata (es. G1, G2, etc.): ")

if not nome_giornata_prefix:
    print("🛑 Prefisso nullo. Operazione annullata.")
else:
    print(f"Inizio aggiornamento classifica con prefisso '{nome_giornata_prefix}'...")
    try:
        # 5.2. Prepara i punteggi della giornata corrente
        cols_da_tenere = ['Email', 'Utente', 'Nickname', 'Punteggio', 'Costo', 'Status', 'Squadra']
        df_giornata_nuova = df_risultati[cols_da_tenere].copy()

        df_giornata_nuova['Email'] = df_giornata_nuova['Email'].astype(str).str.strip().str.lower()

        colonne_rinominate = {
            'Punteggio': f"{nome_giornata_prefix}_Punteggio",
            'Costo': f"{nome_giornata_prefix}_Costo",
            'Status': f"{nome_giornata_prefix}_Status",
            'Squadra': f"{nome_giornata_prefix}_Squadra"
        }
        df_giornata_nuova.rename(columns=colonne_rinominate, inplace=True)
        df_giornata_nuova.set_index('Email', inplace=True)

        # 5.3. Leggi la classifica storica esistente (COME TESTO)
        classifica_ws = ss.worksheet("Classifica")

        # --- FIX: Torniamo a get_all_values() per leggere tutto come TESTO ---
        sheet_data_values = classifica_ws.get_all_values()

        df_storica = pd.DataFrame()

        if len(sheet_data_values) > 1:
            print("Letta classifica storica esistente.")
            header = sheet_data_values[0]
            data = sheet_data_values[1:]
            df_storica = pd.DataFrame(data, columns=header)
            # --- FINE FIX ---

            if 'Email' not in df_storica.columns:
                 raise ValueError("La classifica storica non ha la colonna 'Email'.")

            df_storica['Email'] = df_storica['Email'].astype(str).str.strip().str.lower()
            df_storica = df_storica.set_index('Email')

            colonne_orfane = ['Punteggio', 'Costo', 'Status', 'Squadra']
            df_storica = df_storica.drop(columns=[col for col in colonne_orfane if col in df_storica.columns], errors='ignore')

            colonna_check = f"{nome_giornata_prefix}_Punteggio"
            if colonna_check in df_storica.columns:
                print(f"⚠️ ATTENZIONE: Le colonne con prefisso '{nome_giornata_prefix}' esistono già.")
                sovrascrivi = input("Vuoi sovrascrivere i dati? (sì/no): ")
                if sovrascrivi.lower() != 'sì':
                    raise Exception("Operazione annullata dall'utente.")
                else:
                    cols_da_rimuovere = [col for col in df_storica.columns if col.startswith(nome_giornata_prefix)]
                    df_storica = df_storica.drop(columns=cols_da_rimuovere)
        else:
            print("Foglio 'Classifica' vuoto. Questa è la prima giornata.")

        # 5.4. Unisci la classifica vecchia e i punteggi nuovi
        df_aggiornata = df_storica.join(df_giornata_nuova, how='outer')

        # --- 5.5. Pulisci e Ricalcola il Totale (PRIMA DI TUTTO) ---
        # Questo blocco ora viene eseguito sul DataFrame "stabile", prima che venga "avvelenato"
        if 'Punteggio_Totale' in df_aggiornata.columns:
            df_aggiornata = df_aggiornata.drop(columns=['Punteggio_Totale'])

        colonne_punteggi = [col for col in df_aggiornata.columns if col.endswith('_Punteggio')]
        print(f"Calcolo totale basato su: {colonne_punteggi}")

        punteggio_totale_calcolato = pd.Series(0.0, index=df_aggiornata.index)

        for col in colonne_punteggi:
            # --- FIX: Sostituisci la virgola E converti ---
            col_stringa_pulita = df_aggiornata[col].astype(str).str.replace(',', '.')
            col_numerica = pd.to_numeric(col_stringa_pulita, errors='coerce').fillna(0)
            # --- FINE FIX ---

            df_aggiornata[col] = col_numerica
            punteggio_totale_calcolato = punteggio_totale_calcolato.add(col_numerica, fill_value=0)

        df_aggiornata['Punteggio_Totale'] = punteggio_totale_calcolato

        # --- 5.6. Logica di Overwrite Nomi (DOPO CHE I CALCOLI SONO FINITI) ---
        # Ora questa logica può "avvelenare" il DataFrame, ma non ci interessa più
        df_aggiornata['Utente_Display'] = df_aggiornata['Utente'].fillna(df_aggiornata.get('Utente_Display', ''))
        df_aggiornata['Nickname_Display'] = df_aggiornata['Nickname'].fillna(df_aggiornata.get('Nickname_Display', ''))

        df_aggiornata = df_aggiornata.drop(columns=['Utente', 'Nickname'], errors='ignore')

        # 5.7. Formatta e Ordina
        df_aggiornata['Punteggio_Totale'] = df_aggiornata['Punteggio_Totale'].round(2)
        df_aggiornata.sort_values(by='Punteggio_Totale', ascending=False, inplace=True)

        cols_finali = ['Punteggio_Totale', 'Utente_Display', 'Nickname_Display']
        altre_colonne = sorted([col for col in df_aggiornata.columns if col not in cols_finali])
        cols_finali.extend(altre_colonne)

        df_aggiornata = df_aggiornata[cols_finali]

        colonne_testo = [col for col in df_aggiornata.columns if col.endswith('_Status') or col.endswith('_Squadra')]
        df_aggiornata[colonne_testo] = df_aggiornata[colonne_testo].fillna('N/D')

        df_aggiornata = df_aggiornata.fillna(0)

        df_aggiornata.reset_index(inplace=True)
        df_aggiornata.rename(columns={'index': 'Email'}, inplace=True)

        # --- Versione Ultra-Safe del tuo filtro ---
        df_aggiornata = df_aggiornata[df_aggiornata['Email'].astype(str).str.strip().fillna('') != ""].copy()
        df_aggiornata = df_aggiornata[df_aggiornata['Nickname_Display'].astype(str).str.strip().fillna('') != ""].copy()

        # 5.8. Scrivi sul foglio Google
        print("Scrittura della classifica aggiornata in corso...")
        classifica_ws.clear()

        colonne_output = ['Email', 'Punteggio_Totale', 'Utente_Display', 'Nickname_Display'] + altre_colonne
        df_aggiornata = df_aggiornata[colonne_output]

        output_data = [df_aggiornata.columns.values.tolist()] + df_aggiornata.values.tolist()

        classifica_ws.update(range_name='A1', values=output_data, value_input_option='USER_ENTERED')

        print(f"\n✅ Classifica aggiornata con successo con i dati di '{nome_giornata_prefix}'!")
        print(f"Link: {ss.url}")

    except Exception as e:
        print(f"\n🛑 ERRORE durante l'aggiornamento della classifica:")
        print(e)

# 6. Genera file HTML

In [21]:
# --- 6. GENERATORE CLASSIFICA HTML (Versione Finale con Header Sito) ---

from google.colab import files
import datetime

if 'df_aggiornata' not in locals():
    print("🛑 ERRORE: Esegui prima la Cella 5.")
else:
    print("Generazione HTML con header originale del sito...")

    # Prepariamo i dati
    df_html = df_aggiornata.sort_values(by='Punteggio_Totale', ascending=False).reset_index(drop=True)
    now = datetime.datetime.now().strftime("%d/%m/%Y %H:%M:%S")

    # CSS specifico per la classifica
    html_styles_fanta = """
    <style>
        .fanta-container { max-width: 900px; margin: 40px auto; background: #fff; border-radius: 12px; box-shadow: 0 4px 20px rgba(0,0,0,0.1); overflow: hidden; }
        .fanta-title { text-align: center; padding: 20px; color: #333; margin: 0; background: #f8f9fa; border-bottom: 2px solid #eee; font-weight: 800; }

        ol.fanta-list { list-style: none; padding: 0; margin: 0; }
        .fanta-item { display: flex; align-items: center; padding: 15px 25px; border-bottom: 1px solid #eee; }
        .fanta-item:last-child { border-bottom: none; }

        .pos { font-weight: bold; font-size: 1.4em; min-width: 50px; text-align: center; }
        .info { flex-grow: 1; margin-left: 15px; }
        .nick { font-weight: 700; color: #111; display: block; font-size: 1.1em; }
        .user { font-size: 0.8em; color: #888; text-transform: uppercase; }
        .pts { font-weight: 800; font-size: 1.2em; color: #007bff; background: #eef6ff; padding: 5px 15px; border-radius: 20px; }

        footer.fanta-footer { text-align: center; padding: 20px; font-size: 0.8em; color: #777; margin-bottom: 40px; }
    </style>
    """

    # Generazione righe classifica
    html_rows = ""
    for i, riga in df_html.iterrows():
        pos_num = i + 1
        if pos_num == 1: display_pos = "🥇"
        elif pos_num == 2: display_pos = "🥈"
        elif pos_num == 3: display_pos = "🥉"
        else: display_pos = f"{pos_num}."

        nick = riga.get('Nickname_Display', 'N/D')
        user = riga.get('Utente_Display', 'N/D')
        pts = riga.get('Punteggio_Totale', 0)

        html_rows += f"""
            <li class="fanta-item">
                <span class="pos">{display_pos}</span>
                <div class="info">
                    <span class="nick">{nick}</span>
                    <span class="user">{user}</span>
                </div>
                <span class="pts">{pts}</span>
            </li>
        """

    # Assemblaggio HTML con il TUO Header originale
    html_final = f"""<!DOCTYPE html>
<html lang="it">
<head>
    <link rel="icon" type="image/jpeg" href="img/logo.jpeg">
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Classifica Torneo (CY)</title>
    <link rel="stylesheet" href="css/style.css">
    {html_styles_fanta}
</head>
<body>

    <header>
        <div class="container header-content">
            <a href="index.html">
                <img src="img/logo.jpeg" alt="CY-LEAGUE Logo" class="logo-img">
            </a>
        </div>
        <nav>
                <a href="index.html">Classifica</a>
                <a href="partite.html">Partite</a>
                <a href="marcatori.html">Marcatori</a>
                <a href="squadre.html">Squadre</a>
                <a href="fantacalcio.html">Fantacalcio</a>
        </nav>
    </header>

    <main>
        <div class="fanta-container">
            <h2 class="fanta-title">CLASSIFICA FANTACALCIO</h2>
            <ol class="fanta-list">
                {html_rows}
            </ol>
        </div>
    </main>

    <footer class="fanta-footer">
        Dati aggiornati al: {now}
    </footer>

</body>
</html>
"""

    file_name = "fantacalcio.html"
    with open(file_name, "w", encoding="utf-8") as f:
        f.write(html_final)

    print(f"✅ File '{file_name}' pronto per GitHub!")
    files.download(file_name)

Generazione HTML con header originale del sito...
✅ File 'fantacalcio.html' pronto per GitHub!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>